# 02 - Continual learning with replay and incremental polynomial expansion

Experiences are processed one at a time. At experience *t* the learner may use only the current experience's training
samples, its replay buffer (training samples of earlier experiences), and its current parameters. Evaluation samples are
never used for learning or replay. The polynomial degree grows by one per experience (default), following the **expansion
invariant**: old coefficients are copied exactly, new coefficients start at zero, and `f_{d+1}(x) = f_d(x)` is verified (scores and
probabilities, on training, evaluation, random and out-of-region inputs) before any training.

Nothing here assumes the method works. Compare the strategies at the end before drawing any conclusion.

## Configuration

In [ ]:
SEED = 0
N_TRAIN, N_TEST, NOISE = 200, 500, 0.0
STRATEGY = "ordinary"  # ordinary_no_replay | ordinary | frozen_old | small_old_lr | strategy_b_all | strategy_b_new
STOP_AFTER = (
    None  # e.g. 3 stops after experience 3 so intermediate behaviour can be inspected
)
DEGREES = None  # None -> 1, 2, ..., 10 ; or give an explicit non-decreasing list
LR, EPOCHS = 0.5, 300
M_CURRENT, M_REPLAY, CAPACITY, REPLACE = (
    100,
    100,
    100,
    False,
)  # balanced replay: m per experience
SHOW_STEPS = [
    3,
    6,
    10,
]  # experiences (beyond 1 and 2, shown above) for which the before/after figure is drawn

In [ ]:
import importlib.util
import pathlib
import sys

if importlib.util.find_spec("polynomial_cl") is None:  # not pip-installed: use src/
    sys.path.insert(0, str(pathlib.Path("..").resolve() / "src"))
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams.update({"figure.dpi": 80})
from polynomial_cl import (
    STRATEGIES,
    make_config,
    make_experiences,
    metrics,
    run_continual,
    run_seeds,
)
from polynomial_cl import plotting as P

In [ ]:
exps = make_experiences(seed=SEED, n_train=N_TRAIN, n_test=N_TEST, noise=NOISE)
cfg = make_config(
    STRATEGY,
    lr=LR,
    epochs=EPOCHS,
    m_current=M_CURRENT,
    m_replay=M_REPLAY,
    capacity=CAPACITY,
    replace=REPLACE,
    seed=SEED,
    stop_after=STOP_AFTER,
    degrees=DEGREES,
)
fig, axs = plt.subplots(1, 2, figsize=(9, 4.2))
P.plot_ground_truth(axs[0], title="Ground truth")
P.plot_ground_truth(axs[1], title="Experiences = vertical strips (training samples)")
for e in exps:
    P.plot_samples(axs[1], e.train.x, e.train.y, s=6)
    axs[1].axvline(e.window[0], color="k", lw=0.3)
plt.tight_layout()
plt.show()
result = run_continual(exps, cfg)
print(cfg)

## Stage A - initial model (experience 1)

In [ ]:
r1 = result.records[0]
print("degree:", r1.degree_after, "| features:", r1.feature_names)
print(f"training loss: {r1.loss_history[0]:.4f} -> {r1.loss_history[-1]:.4f}")
print(f"accuracy on experience 1 (fixed test set): {r1.acc_current:.3f}")
stage_a = {
    "scores": r1.model_after.score(exps[0].test.x),
    "labels": r1.model_after.predict(exps[0].test.x),
    "coefficients": r1.model_after.w.copy(),
}  # saved for later comparison
print(P.format_coefficient_table(r1))
fig = P.plot_step(r1, exps)
plt.show()

## Stage B - experience 2: expand, verify the invariant, then train

The second panel (right after expansion, no training) must be identical to the first.

In [ ]:
r2 = result.records[1]
print(
    f"expansion degree {r2.degree_before} -> {r2.degree_after}:  max|score diff| = {r2.invariant_score_diff:.2e},  max|proba diff| = {r2.invariant_proba_diff:.2e}"
)
print("replay batch composition (samples drawn per experience):", r2.batch_counts)
print(P.format_coefficient_table(r2))
P.plot_step(r2, exps)
plt.show()
P.plot_coefficients(r2)
plt.show()

## Stage C - experiences 3 ... 10

In [ ]:
for t in SHOW_STEPS:
    if t - 1 < len(result.records):
        r = result.records[t - 1]
        print(
            f"--- experience {t}: degree {r.degree_before} -> {r.degree_after}, batch {r.batch_counts}"
        )
        print(
            f"    invariant: score diff {r.invariant_score_diff:.1e}, proba diff {r.invariant_proba_diff:.1e}"
        )
        P.plot_step(r, exps)
        plt.show()
        P.plot_coefficients(r)
        plt.show()

## Learning history and compact summary

In [ ]:
P.plot_learning_history(result)
plt.show()
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
P.plot_accuracy_matrix(result, axs[0])
last = result.records[-1].model_after
x_te = np.concatenate([e.test.x for e in exps[: len(result.records)]])
y_te = np.concatenate([e.test.y for e in exps[: len(result.records)]])
P.plot_correctness(
    last,
    axs[1],
    samples=(x_te, y_te),
    title="Final model: correct/incorrect regions + all evaluation samples",
)
plt.tight_layout()
plt.show()
print(metrics.format_table(metrics.summary_table(result)))

## Strategy comparison (single seed, identical data, evaluation sets, learning rate, epochs and replay budget)

| name | what changes |
|---|---|
| `ordinary_no_replay` | one global learning rate, **no replay** (isolates the effect of replay) |
| `ordinary` | one global learning rate, replay |
| `frozen_old` | existing coefficients frozen; only newly introduced coefficients train |
| `small_old_lr` | existing coefficients use 0.1 x the learning rate |
| `strategy_b_all` | `f = f_old + g` with the constraint `g(x_ref) = 0` on reference inputs (3 buffered inputs per old experience); g over all coefficients |
| `strategy_b_new` | as above, g only over new coefficients |

Preserving `f_old` at a finite set of reference points does **not** preserve it elsewhere; the columns below measure that directly.

In [ ]:
names = list(STRATEGIES)


def cfg_for(n, **kw):
    shared = {
        "lr": LR,
        "epochs": EPOCHS,
        "m_current": M_CURRENT,
        "m_replay": M_REPLAY,
        "capacity": CAPACITY,
        "replace": REPLACE,
    }
    # A preset owns its defining parameters; shared settings only fill the rest.
    shared = {k: v for k, v in shared.items() if k not in STRATEGIES[n]}
    return make_config(n, **shared, **kw)


runs = {n: run_continual(exps, cfg_for(n, seed=SEED)) for n in names}
rows = []
for n, r in runs.items():
    last = r.records[-1]
    rows.append(
        {
            "strategy": n,
            "final_mean_acc": last.mean_acc,
            "final_acc_current": last.acc_current,
            "final_acc_E1": r.acc_matrix[-1, 0],
            "final_mean_forgetting": last.mean_forgetting,
            "final_train_loss": float(last.loss_history[-1]),
            "loss_reduction_last_exp": float(
                last.loss_history[0] - last.loss_history[-1]
            ),
            "trainable_dims_last_exp": last.n_trainable,
            "trainable_l1_update_last_exp": last.trainable_l1_update,
            "max_old_score_change_last_exp": last.max_old_score_change,
            "old_flip_fraction_last_exp": last.old_flip_fraction,
            # null_dim is meaningful only for Strategy B's constrained update.
            "strategy_b_null_dim": last.null_dim
            if last.null_dim is not None
            else "n/a",
            "max_frozen_change": max(
                (
                    x.frozen_max_change
                    for x in r.records
                    if x.frozen_max_change is not None
                ),
                default=float("nan"),
            ),
        }
    )
print(metrics.format_table(rows))

fig, ax = plt.subplots(figsize=(8, 4.5))
for n, r in runs.items():
    ax.plot(
        range(1, len(r.records) + 1),
        [x.mean_acc for x in r.records],
        marker="o",
        ms=3,
        label=n,
    )
ax.set_xlabel("Experience")
ax.set_ylabel("Mean accuracy on observed experiences")
ax.set_title("Strategy comparison: accuracy and learning behavior")
ax.legend(fontsize=7)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for n, r in runs.items():
    axes[0].plot(
        range(1, len(r.records) + 1),
        [float(x.loss_history[0] - x.loss_history[-1]) for x in r.records],
        marker="o",
        ms=3,
        label=n,
    )
    axes[1].plot(
        range(1, len(r.records) + 1),
        [x.trainable_l1_update for x in r.records],
        marker="o",
        ms=3,
        label=n,
    )
axes[0].set_title("Loss reduction within each experience")
axes[0].set_xlabel("Experience")
axes[0].set_ylabel("Initial loss − final loss")
axes[1].set_title("Magnitude of trainable parameter updates")
axes[1].set_xlabel("Experience")
axes[1].set_ylabel("L1 update")
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

### Ablation: does the incremental degree schedule matter?

Three conditions on the same experiences (paired by seed) with identical replay, epochs and optimiser, five seeds:
degree 1 throughout (no expansion); growing 1, 2, ..., 10; and degree 10 from experience 1 (same final capacity,
no gradual growth). *Growing vs degree 1* asks whether added capacity helps at all; *growing vs degree 10* asks
whether adding it gradually matters. The same experiment is available as
`python experiments/synthetic_degree_ablation.py`.

In [ ]:
# Paired-seed ablation: does expansion help versus no expansion, and does timing matter?
# All conditions reuse the same generated experience stream within each seed.
abl = {}
degree_conditions = (
    ("fixed degree 1 (no expansion)", [1] * len(exps)),
    ("grow 1..10", None),
    ("fixed degree 10 (capacity control)", [10] * len(exps)),
)
for label, deg in degree_conditions:
    rs = run_seeds(
        [0, 1, 2, 3, 4],
        cfg_for("ordinary", degrees=deg),
        {"n_train": N_TRAIN, "n_test": N_TEST, "noise": NOISE},
    )
    abl[label] = rs
    final_acc = np.asarray([r.records[-1].mean_acc for r in rs])
    final_forgetting = np.asarray([r.records[-1].mean_forgetting for r in rs])
    print(
        f"{label:34s} "
        f"final mean accuracy {final_acc.mean():.3f} +/- {final_acc.std(ddof=1):.3f} | "
        f"final mean forgetting {final_forgetting.mean():.3f} +/- "
        f"{final_forgetting.std(ddof=1):.3f}"
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for label, rs in abl.items():
    per_seed_acc = np.asarray([[step.mean_acc for step in r.records] for r in rs])
    per_seed_forgetting = np.asarray(
        [[step.mean_forgetting for step in r.records] for r in rs]
    )
    x = np.arange(1, per_seed_acc.shape[1] + 1)
    axes[0].plot(x, per_seed_acc.mean(axis=0), marker="o", ms=3, label=label)
    axes[0].fill_between(
        x,
        per_seed_acc.mean(axis=0) - per_seed_acc.std(axis=0, ddof=1),
        per_seed_acc.mean(axis=0) + per_seed_acc.std(axis=0, ddof=1),
        alpha=0.15,
    )
    axes[1].plot(x, per_seed_forgetting.mean(axis=0), marker="o", ms=3, label=label)
axes[0].set_title("Mean accuracy across experiences")
axes[0].set_xlabel("Experience")
axes[0].set_ylabel("Mean accuracy")
axes[1].set_title("Mean forgetting across experiences")
axes[1].set_xlabel("Experience")
axes[1].set_ylabel("Mean forgetting")
axes[0].legend(fontsize=7)
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

print("Interpretation:")
print(
    "- expanding vs fixed degree 1 tests whether adding polynomial capacity helps at all."
)
print(
    "- expanding vs fixed degree 10 tests whether gradual availability helps versus final capacity from the start."
)

### Strategy B inspection

`ref_score_change` (change at the reference inputs) should be ~1e-15 by construction. `old_score_change` is measured on the
*whole* old evaluation sets, which are not the reference inputs. `free dims` is specific to Strategy B and is not a general learning indicator. For every strategy, inspect the loss history, `loss_reduction`, and `trainable_l1_update`; `null_dim=None` means no null-space constraint was applied, not that learning did not happen. A zero free dimension in Strategy B means g cannot update.

In [ ]:
for n in ("strategy_b_all", "strategy_b_new"):
    print("==", n)
    print(
        metrics.format_table(
            [
                {
                    "exp": r.t + 1,
                    "degree": r.degree_after,
                    "null_dim": r.null_dim
                    if r.null_dim is not None
                    else "n/a (unconstrained)",
                    "trainable_dims": r.n_trainable,
                    "loss_reduction": float(r.loss_history[0] - r.loss_history[-1]),
                    "trainable_l1_update": r.trainable_l1_update,
                    "acc_current": r.acc_current,
                    "mean_acc": r.mean_acc,
                    "ref_score_change": r.ref_score_change
                    if r.ref_score_change is not None
                    else float("nan"),
                    "old_score_change(eval sets)": r.max_old_score_change,
                    "old_flip_frac": r.old_flip_fraction,
                    "frozen_change": r.frozen_max_change
                    if r.frozen_max_change is not None
                    else float("nan"),
                }
                for r in runs[n].records
            ],
            "{:.3g}",
        )
    )
    P.plot_learning_history(runs[n])
    plt.show()
    P.plot_step(runs[n].records[-1], exps)
    plt.show()

print("A null dimension is a constraint diagnostic, not a general learning indicator.")
print(
    "For unconstrained strategies, inspect loss reduction, trainable updates, accuracy, and forgetting instead."
)

# Compare the final binary decision boundaries on the same domain and test samples.
# "ordinary" is the distinct preset equivalent to the former unconstrained Strategy A-all.
comparison_names = (
    ("ordinary (former A-all behavior)", "ordinary"),
    ("frozen_old (former A-new behavior)", "frozen_old"),
    ("Strategy B: all coefficients", "strategy_b_all"),
    ("Strategy B: new coefficients", "strategy_b_new"),
)
x_eval = np.concatenate([e.test.x for e in exps[: len(result.records)]])
y_eval = np.concatenate([e.test.y for e in exps[: len(result.records)]])
fig, axes = plt.subplots(1, len(comparison_names), figsize=(20, 4.5))
for ax, (label, strategy_name) in zip(axes, comparison_names):
    P.plot_correctness(
        runs[strategy_name].records[-1].model_after,
        ax=ax,
        samples=(x_eval, y_eval),
        title=label,
        legend=False,
    )
plt.tight_layout()
plt.show()

## Multiple seeds

Seeds change the training samples, replay selection and shuffling (evaluation sets also change with the seed but are
fixed within a seed). Values are mean +/- standard deviation over seeds.

In [ ]:
SEEDS = [0, 1, 2, 3, 4]


def mean_std(runs_, getter):
    vals = [getter(r) for r in runs_]
    return np.mean(vals), np.std(vals)


rows = []
for n in names:
    base = cfg_for(n)
    rs = run_seeds(SEEDS, base, {"n_train": N_TRAIN, "n_test": N_TEST, "noise": NOISE})
    m1, s1 = mean_std(rs, lambda r: r.records[-1].mean_acc)
    m2, s2 = mean_std(rs, lambda r: r.records[-1].acc_current)
    m3, s3 = mean_std(rs, lambda r: r.records[-1].mean_forgetting)
    m4, s4 = mean_std(rs, lambda r: r.acc_matrix[-1, 0])
    rows.append(
        {
            "strategy": n,
            "final_mean_acc": f"{m1:.3f}+/-{s1:.3f}",
            "final_acc_current": f"{m2:.3f}+/-{s2:.3f}",
            "final_acc_E1": f"{m4:.3f}+/-{s4:.3f}",
            "final_mean_forgetting": f"{m3:.3f}+/-{s3:.3f}",
        }
    )
print(metrics.format_table(rows))

## Why does Strategy B fit the sine curve worse? Reference points, not optimisation

Strategy B pins `g(x_ref) = 0` at 3 buffered inputs per old experience. `null_dim` grows with the degree, yet from
experience 6 on the loss barely decreases (`loss_reduction` ~ 1e-3 above). Two explanations are tested on five seeds:
*too little optimisation* (10x epochs) versus *the constraint itself* (fewer or more reference points per old
experience). The last columns measure how much the old function actually moves on the old evaluation sets.

In [ ]:
import dataclasses

variants = [
    ("ordinary", "ordinary", {}, EPOCHS),
    ("strategy_b_all, 1 ref/exp", "strategy_b_all", {"n_ref_per_exp": 1}, EPOCHS),
    ("strategy_b_all, 2 ref/exp", "strategy_b_all", {"n_ref_per_exp": 2}, EPOCHS),
    ("strategy_b_all, 3 ref/exp (default)", "strategy_b_all", {}, EPOCHS),
    ("strategy_b_all, 5 ref/exp", "strategy_b_all", {"n_ref_per_exp": 5}, EPOCHS),
    ("strategy_b_all, 3 ref/exp, 10x epochs", "strategy_b_all", {}, 10 * EPOCHS),
    ("frozen_old", "frozen_old", {}, EPOCHS),
]
rows = []
for label, name, kw, epochs in variants:
    cfg_v = dataclasses.replace(cfg_for(name, **kw), epochs=epochs)
    rs = run_seeds(SEEDS, cfg_v, {"n_train": N_TRAIN, "n_test": N_TEST, "noise": NOISE})
    final = np.array([r.records[-1].mean_acc for r in rs])
    rows.append(
        {
            "variant": label,
            "final_mean_acc": f"{final.mean():.3f}+/-{final.std(ddof=1):.3f}",
            "mean_acc_current_exp": f"{np.mean([[x.acc_current for x in r.records] for r in rs]):.3f}",
            "final_forgetting": f"{np.mean([r.records[-1].mean_forgetting for r in rs]):.4f}",
            "mean_old_score_change": f"{np.mean([[x.max_old_score_change for x in r.records[1:]] for r in rs]):.3f}",
            "null_dim_last": rs[0].records[-1].null_dim
            if rs[0].records[-1].null_dim is not None
            else "n/a",
        }
    )
print(metrics.format_table(rows))

## Per-coefficient learning rates

`lr_old_factor` already gives old and new coefficients different rates; `lr_feature_power = p` additionally gives
coefficient *i* the rate `lr * (max_rms / rms_i) ** p`, so the small high-degree features take larger steps
(p = 1 is diagonal preconditioning). 300 epochs with a shared rate are far from converged here (0.967 vs 0.991 after
10,000), so the rate allocation matters. The comparison below repeats the expansion question with both settings.

In [ ]:
setups = {
    "shared rate (p=0, lr=0.5)": {"lr": 0.5, "lr_feature_power": 0.0},
    "per-feature rate (p=1, lr=4)": {"lr": 4.0, "lr_feature_power": 1.0},
}
conditions = {
    "degree 1 (no expansion)": [1] * 10,
    "grow 1..10": None,
    "fixed degree 10": [10] * 10,
}
rows = []
for cond, degrees in conditions.items():
    row = {"condition": cond}
    for label, kw in setups.items():
        rs = run_seeds(
            SEEDS,
            make_config("ordinary", epochs=EPOCHS, degrees=degrees, **kw),
            {"n_train": N_TRAIN, "n_test": N_TEST, "noise": NOISE},
        )
        acc = np.array([r.records[-1].mean_acc for r in rs])
        fgt = np.mean([r.records[-1].mean_forgetting for r in rs])
        row[label] = f"{acc.mean():.3f}+/-{acc.std(ddof=1):.3f} (forgetting {fgt:.4f})"
    rows.append(row)
print(metrics.format_table(rows))